# Foundations of Multi-Agents systems and CrewAI

In [40]:
# warning control
import warnings
warnings.filterwarnings("ignore")

In [2]:
from crewai import Agent, Task, Crew
from dotenv import load_dotenv
import yaml
import os

c:\Users\Pichau\OneDrive\Área de Trabalho\Estagio\Curso MultiAgentes\venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:775: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warn(


In [3]:
with open("config.yaml", "r") as config_file:
  config = yaml.safe_load(config_file)
os.environ["OPENAI_API_KEY"] = config.get("OPENAI_API_KEY")

In [4]:
os.environ["OPENAI_MODEL_NAME"] = "gpt-3.5-turbo"

# Creating Agents

* Defina seus agentes e forneça a eles uma função, um objetivo e uma história de fundo.
* Foi constatado que os LLMs têm melhor desempenho quando desempenham papéis.

Agente: Planner

Nota: A vantagem de usar várias strings:

* varname = "linha 1 do texto"
          "linha 2 do texto"
versus a doutrina de aspas triplas:

* varname = """linha 1 do texto
             linha 2 do texto
          """
          
é que ele pode evitar a adição de espaços em branco e caracteres de nova linha, tornando-o melhor formatado para ser passado ao LLM.


In [43]:
planner = Agent(
    role="Planejador de Conteúdo",
    goal="Planejar conteúdo envolvente e factualmente preciso sobre {topic}",
    backstory="Você está trabalhando no planejamento de um artigo de blog "
              "sobre o tema: {topic}."
              "Você coleta informações que ajudam o público a "
              "aprender algo e tomar decisões informadas. "
              "Seu trabalho é a base para que o Redator de Conteúdo "
              "escreva um artigo sobre esse tema.",
    allow_delegation=False,
    verbose=True
)

LLM value is None


Agente: Writer

In [44]:
writer = Agent(
    role="Redator de Conteúdo",
    goal="Escrever um artigo opinativo perspicaz e factualmente preciso "
         "sobre o tema: {topic}",
    backstory="Você está trabalhando na redação "
              "de um novo artigo opinativo sobre o tema: {topic}. "
              "Você baseia sua escrita no trabalho do "
              "Planejador de Conteúdo, que fornece um esboço "
              "e contexto relevante sobre o tema. "
              "Você segue os principais objetivos e a "
              "direção do esboço, conforme fornecido pelo Planejador de Conteúdo. "
              "Você também fornece insights objetivos e imparciais "
              "e os embasa com as informações "
              "fornecidas pelo Planejador de Conteúdo. "
              "No seu artigo opinativo, você reconhece "
              "quando suas declarações são opiniões, "
              "em vez de afirmações objetivas.",
    allow_delegation=False,
    verbose=True
)


LLM value is None


Agent: Editor

In [45]:
editor = Agent(
    role="Editor",
    goal="Editar um post de blog para alinhar com "
         "o estilo de escrita da organização.",
    backstory="Você é um editor que recebe um post de blog "
              "do Redator de Conteúdo. "
              "Seu objetivo é revisar o post "
              "para garantir que ele siga as melhores práticas jornalísticas, "
              "forneça pontos de vista equilibrados "
              "ao apresentar opiniões ou afirmações, "
              "e também evite tópicos ou opiniões controversas "
              "sempre que possível.",
    allow_delegation=False,
    verbose=True
)

LLM value is None


# Creating Tasks
* Defina suas tarefas e forneça a elas uma descrição, saída esperada e agente.

Task: Plan

In [46]:
plan = Task(
    description=(
        "1. Priorize as últimas tendências, principais players "
            "e notícias relevantes sobre {topic}.\n"
        "2. Identifique o público-alvo, considerando "
            "seus interesses e pontos de dor.\n"
        "3. Desenvolva um esboço de conteúdo detalhado, incluindo "
            "uma introdução, pontos principais e um call to action.\n"
        "4. Inclua palavras-chave de SEO e dados ou fontes relevantes."
    ),
    expected_output="Um documento de plano de conteúdo abrangente "
        "com um esboço, análise de público-alvo, "
        "palavras-chave de SEO e recursos.",
    agent=planner,
)

Task: Write

In [47]:
write = Task(
    description=(
        "1. Use o plano de conteúdo para criar um "
            "post de blog atraente sobre {topic}.\n"
        "2. Incorpore as palavras-chave de SEO naturalmente.\n"
        "3. As seções/subtítulos devem ser devidamente nomeados "
            "de forma envolvente.\n"
        "4. Certifique-se de que o post esteja estruturado com uma "
            "introdução envolvente, um corpo perspicaz "
            "e uma conclusão resumida.\n"
        "5. Faça uma revisão para corrigir erros gramaticais e "
            "garantir o alinhamento com o tom de voz da marca.\n"
    ),
    expected_output="Um post de blog bem escrito "
        "em formato markdown, pronto para publicação, "
        "cada seção deve ter 2 ou 3 parágrafos.",
    agent=writer,
)

Task: Edit

In [48]:
edit = Task(
    description=("Revise o post de blog fornecido para "
                 "corrigir erros gramaticais e "
                 "garantir o alinhamento com o tom de voz da marca."),
    expected_output="Um post de blog bem escrito em formato markdown, "
                    "pronto para publicação, "
                    "cada seção deve ter 2 ou 3 parágrafos.",
    agent=editor
)

# Creating the Crew
* Crie sua equipe de Agentes
* Passe as tarefas a serem executadas por esses agentes.
    - Nota: Para este exemplo simples, as tarefas serão executadas sequencialmente (ou seja, são dependentes umas das outras), portanto a ordem da tarefa na lista é importante.
* verbose=2 permite que você veja todos os logs da execução

In [49]:
crew = Crew(
  agents=[planner, writer, editor],
  tasks=[plan, write, edit],
  verbose=True
)

Overriding of current TracerProvider is not allowed


# Running the Crew
Observação: os LLMs podem fornecer saídas diferentes para a mesma entrada, portanto, o que você obtém pode ser diferente do que vê no vídeo.

In [50]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

# Agent: Planejador de Conteúdo
## Task: 1. Priorize as últimas tendências, principais players e notícias relevantes sobre Artificial Intelligence.
2. Identifique o público-alvo, considerando seus interesses e pontos de dor.
3. Desenvolva um esboço de conteúdo detalhado, incluindo uma introdução, pontos principais e um call to action.
4. Inclua palavras-chave de SEO e dados ou fontes relevantes.


# Agent: Planejador de Conteúdo
## Final Answer: 
**Plano de Conteúdo sobre Inteligência Artificial**

**1. Últimas tendências, principais players e notícias relevantes sobre Inteligência Artificial:**

- **Tendências:**
  - Crescimento de modelos de linguagem, como GPT-4 e BERT, que promovem a comunicação natural entre humanos e máquinas.
  - Avanços em aprendizado de máquina explicado, aumentando a transparência dos algoritmos.
  - A integração de IA em setores variados, como saúde, finanças, educação e entretenimento.
  - Desenvolvimento de IA ética e responsável, que busca minimizar preco

In [ ]:
from IPython.display import Markdown
Markdown(str(result))

# A Revolução da Inteligência Artificial: Tendências, Desafios e Futuro

Nos últimos anos, a Inteligência Artificial (IA) emergiu como uma das tecnologias mais transformadoras da nossa era. Com suas aplicações se expandindo de maneira exponencial, a IA está moldando não apenas o futuro dos negócios, mas também a sociedade como um todo. Neste artigo, exploraremos as últimas tendências em IA, os principais players do setor e como essa revolução pode influenciar seu negócio.

## O que é Inteligência Artificial?

Inteligência Artificial refere-se à simulação de processos de inteligência humana por máquinas, especialmente sistemas computacionais. Isso inclui aprendizado, raciocínio e autocorreção. Embora a IA não seja uma invenção recente, suas raízes remontam à década de 1950, quando os pesquisadores começaram a desenvolver algoritmos que poderiam aprender e se adaptar. Desde então, o campo evoluiu significativamente, com o surgimento de tecnologias como aprendizado de máquina e redes neurais, que permitem que os sistemas processem grandes volumes de dados de forma eficiente.

A crescente popularidade de modelos de linguagem, como GPT-4 e BERT, revolucionou a interação entre humanos e máquinas, tornando a comunicação mais natural e intuitiva. Esses avanços permitem que as máquinas entendam e gerem texto de maneiras que antes pareciam impossíveis, facilitando diversas tarefas e otimizando processos empresariais.

## Tendências atuais em IA

Atualmente, algumas tendências em IA estão se destacando de maneira significativa. O crescimento dos modelos de linguagem é uma delas. Com a capacidade de compreender e gerar uma ampla variedade de expressões linguísticas, esses modelos estão se tornando ferramentas essenciais em áreas como atendimento ao cliente, marketing e até na redação de conteúdo.

Outra tendência importante é a ênfase em uma IA ética e responsável. À medida que as empresas buscam implementar soluções baseadas em IA, há uma preocupação crescente sobre preconceitos e viés nos dados. Essa busca por uma tecnologia mais transparente e justa é fundamental para garantir que a IA beneficie todos, minimizando desigualdades e promovendo um avanço equitativo.

## Principais players no setor de IA

Quando falamos sobre IA, várias empresas se destacam como líderes do setor. Organizações como Google, OpenAI, IBM, Microsoft e Amazon estão na vanguarda dessas inovações. O Google, por meio de iniciativas como DeepMind e Google AI, tem realizado investimentos significativos em modelos avançados de IA, impulsionando o aprendizado de máquina e a análise de dados em tempo real.

A OpenAI ganhou notoriedade pela introdução de modelos de linguagem preditiva, como o GPT-4, que têm atraído a atenção de diversos setores. Da mesma forma, a IBM, com seu Watson, foca em soluções para saúde e comércio, enquanto a Microsoft e a Amazon integram suas plataformas de IA em serviços em nuvem, proporcionando soluções escaláveis que podem beneficiar empresas de todos os portes.

## Desafios e Oportunidades

Apesar do avanço das tecnologias de IA, os desafios permanecem. A integração da IA com sistemas existentes pode apresentar dificuldades, especialmente em ambientes industriais. Além disso, os riscos associados à segurança de dados e questões éticas levantam preocupações que devem ser abordadas.

Por outro lado, esses desafios também representam oportunidades significativas. As empresas que conseguiram integrar a IA em seus processos relatam aumentos notáveis na eficiência e na capacidade de inovação. Ao abordar as questões éticas e implementar soluções responsáveis, as organizações podem criar um ambiente favorável que não apenas atende às demandas do presente, mas também se prepara para um futuro sustentável.

## O Futuro da Inteligência Artificial

Olhando para o futuro da Inteligência Artificial, as projeções indicam um progresso contínuo. Espera-se que a IA tenha um papel central em diversos setores, desde o aprimoramento da educação até a transformação na área da saúde. Com os avanços em automação e análise preditiva, empresas poderão não apenas reagir às mudanças no mercado, mas antecipar as necessidades e demandas de seus clientes.

À medida que a tecnologia evolui, a colaboração entre humanos e máquinas se tornará ainda mais importante. O desafio será encontrar um equilíbrio entre aproveitar as capacidades da IA e manter um enfoque humanocêntrico nas decisões e interações, assegurando que a tecnologia sirva para melhorar a experiência humana.

## Conclusão

A Inteligência Artificial é, sem dúvida, uma ferramenta revolucionária que está remodelando a maneira como vivemos e trabalhamos. Desde suas tendências atuais até os desafios que precisam ser superados, a IA oferece um mundo de possibilidades. Para aqueles prontos para abraçar essa mudança, as oportunidades são ilimitadas.

Se você está interessado em acompanhar as últimas tendências e inovações em IA, não deixe de se inscrever em nossa newsletter e participar do nosso próximo webinar. Vamos explorar juntos o impacto desta tecnologia no futuro dos negócios!

# Oque são Agentes?
* São LLMs que podem falar consigo mesmas.
* Tem o seu processo de pensamento
* Podem receber tarefas e usar ferramentas
para dar respostas muito mais completas doque
fazendo de outra meneira
* Pode ter multiplos agentes conectados entre si
* Um agente pode passar tarefas a outro agente.
* Pode ter agentes especificos que são excelentes em alguma coisa